# Rental Agreement Metadata Extraction

**OCR → clean/prepare text → LLM extraction → rule-based normalization/validation → evaluation**

In [2]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.config import TRAIN_CSV, TRAIN_DIR, TEST_DIR, OUTPUT_DIR, SCHEMA_FIELDS, MODEL_NAME
from app.document_loader import build_document_dataframe
from app.text_preprocessor import prepare_document
from app.extractor import extract_metadata
from app.validation import validate_metadata
from app.evaluator import evaluate, failure_cases

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_NAME)

Project root: /Users/prasad/Downloads/metadata-extraction
Model: qwen2.5:3b


## 1. Load training labels

In [4]:
train_df = pd.read_csv(TRAIN_CSV)
print("Training label rows:", len(train_df))
display(train_df.head())

Training label rows: 10


,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R


## 2. OCR / text extraction

Extract text from DOCX/PDF/images. No LLM is used here.

In [5]:
documents_df = build_document_dataframe(TRAIN_DIR, TEST_DIR)
documents_df.to_csv(OUTPUT_DIR / "extracted_documents.csv", index=False)
print("Number of extracted documents:", len(documents_df))
display(documents_df[["File Name", "Split", "Character Count", "Status"]])

Number of extracted documents: 14


,File Name,Split,Character Count,Status
0,18325926-Rental-Agreement-1.docx,train,2952,success
1,36199312-Rental-Agreement.png,train,2297,success
2,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,train,1081,success
3,46239065-Standard-Rental-Agreement-Rental-With...,train,19571,success
4,47854715-RENTAL-AGREEMENT.docx,train,4558,success
5,50070534-RENTAL-AGREEMENT.docx,train,2677,success
6,54770958-Rental-Agreement.png,train,3419,success
7,54945838-Rental-Agreement.png,train,3484,success
8,6683127-House-Rental-Contract-GERALDINE-GALINA...,train,3192,success
9,6683129-House-Rental-Contract-Geraldine-Galina...,train,16876,success


## 3. Clean / prepare text

Cleaning is conservative: whitespace/formatting noise is reduced, but OCR names, numbers and dates are not silently corrected.

In [6]:
documents_df["Prepared Text"] = documents_df["Text"].map(prepare_document)
display(documents_df[["File Name", "Character Count", "Status"]].head())

,File Name,Character Count,Status
0,18325926-Rental-Agreement-1.docx,2952,success
1,36199312-Rental-Agreement.png,2297,success
2,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,1081,success
3,46239065-Standard-Rental-Agreement-Rental-With...,19571,success
4,47854715-RENTAL-AGREEMENT.docx,4558,success


## 4. LLM extraction

Run only on matched training documents so evaluation is comparable with the supplied labels.

In [16]:
train_docs = documents_df[(documents_df["Split"] == "train") & (documents_df["Status"] == "success")].copy()
label_stems = set(train_df["File Name"].astype(str).str.strip())
train_docs = train_docs[train_docs["File Stem"].isin(label_stems)].copy()
print("Training labels:", len(train_df))
print("Matched training documents:", len(train_docs))

pred_rows = []
for _, row in tqdm(train_docs.iterrows(), total=len(train_docs)):
    try:
        metadata = validate_metadata(extract_metadata(row["Prepared Text"]))
        status = "success"
    except Exception as exc:
        metadata = {field: None for field in SCHEMA_FIELDS}
        status = f"error: {exc}"
    pred_rows.append({"File Name": row["File Stem"], "Split": "train", **metadata, "Extraction Status": status})

predictions_df = pd.DataFrame(pred_rows)
predictions_df.to_csv(OUTPUT_DIR / "train_predictions.csv", index=False)
display(predictions_df)

Training labels: 10
Matched training documents: 9


  0%|          | 0/9 [00:00<?, ?it/s]

,File Name,Split,Agreement Value,Agreement Start Date,Agreement End Date,Renewal Notice (Days),Party One,Party Two,Extraction Status
0,18325926-Rental-Agreement-1,train,4000,05.12.2008,16.12.2009,90,MR.K.Kuttan,P.M. Narayana Namboodri 'Laxmi Leela',success
1,36199312-Rental-Agreement,train,40000,01.05.2010,11.05.2011,30,Balaji.R,Kartheek R,success
2,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,train,9979,NaN,NaN,NaN,M. V. V« VI JAY A SHANKAR,MADDIREDDY BHARGAVA REDDY,success
3,47854715-RENTAL-AGREEMENT,train,9000,01.04.2010,NaN,60,P C MATHEW,L GOPINATH,success
4,50070534-RENTAL-AGREEMENT,train,10000,01.04.2010,30.03.2011,90,P. JohnsonRavikumar,Saravanan BV,success
5,54770958-Rental-Agreement,train,5500,02.04.2011,02.04.2012,60,Mrs.Asha Ramesh & Mr.Ramesh.K.N,Mrs.Sadasivuni Deepti & Mr.Sadasivuni Kiran,success
6,54945838-Rental-Agreement,train,8000,01.04.2011,01.04.2013,90,K. Parthasarathy,Veerabrahmam Bathini,success
7,6683127-House-Rental-Contract-GERALDINE-GALINA...,train,6500,20.05.2007,20.05.2008,3,ANTONIO LEVY S. INGLES. JR. and/or MARY ROSE C...,GERALDINE O. GALINATO,success
8,6683129-House-Rental-Contract-Geraldine-Galina...,train,NaN,NaN,NaN,NaN,NaN,NaN,"error: HTTPConnectionPool(host='localhost', po..."


In [26]:
import re
from datetime import datetime


def validate_metadata(metadata):

    cleaned = {}

    # -----------------------------
    # Agreement Value
    # -----------------------------
    value = metadata.get("Agreement Value")

    if value is not None:
        value = str(value).strip()

        # Keep digits only
        value_digits = re.sub(r"[^\d]", "", value)

        cleaned["Agreement Value"] = (
            value_digits if value_digits else None
        )
    else:
        cleaned["Agreement Value"] = None


    # -----------------------------
    # Start Date
    # -----------------------------
    start_date = metadata.get("Agreement Start Date")

    if start_date:
        start_date = str(start_date).strip()

        try:
            datetime.strptime(start_date, "%d.%m.%Y")
            cleaned["Agreement Start Date"] = start_date
        except ValueError:
            cleaned["Agreement Start Date"] = None
    else:
        cleaned["Agreement Start Date"] = None


    # -----------------------------
    # End Date
    # -----------------------------
    end_date = metadata.get("Agreement End Date")

    if end_date:
        end_date = str(end_date).strip()

        try:
            datetime.strptime(end_date, "%d.%m.%Y")
            cleaned["Agreement End Date"] = end_date
        except ValueError:
            cleaned["Agreement End Date"] = None
    else:
        cleaned["Agreement End Date"] = None


    # -----------------------------
    # Renewal Notice
    # -----------------------------
    notice = metadata.get("Renewal Notice (Days)")

    if notice is not None:
        notice = str(notice).strip()
        notice_digits = re.sub(r"[^\d]", "", notice)

        cleaned["Renewal Notice (Days)"] = (
            notice_digits if notice_digits else None
        )
    else:
        cleaned["Renewal Notice (Days)"] = None


    # -----------------------------
    # Party One
    # -----------------------------
    party_one = metadata.get("Party One")

    if party_one:
        party_one = str(party_one).strip()

        party_one = re.sub(
            r"^(Mr\.|Mrs\.|Ms\.|Prof\.|Sri\.)\s*",
            "",
            party_one,
            flags=re.IGNORECASE
        )

        party_one = re.split(
            r"\s+(?:S/o|D/o|W/o)\b",
            party_one,
            flags=re.IGNORECASE
        )[0].strip()

        cleaned["Party One"] = party_one
    else:
        cleaned["Party One"] = None


    # -----------------------------
    # Party Two
    # -----------------------------
    party_two = metadata.get("Party Two")

    if party_two:
        party_two = str(party_two).strip()

        party_two = re.sub(
            r"^(Mr\.|Mrs\.|Ms\.|Prof\.|Sri\.)\s*",
            "",
            party_two,
            flags=re.IGNORECASE
        )

        party_two = re.split(
            r"\s+(?:S/o|D/o|W/o)\b",
            party_two,
            flags=re.IGNORECASE
        )[0].strip()

        # Remove quoted property/business name
        party_two = re.sub(
            r"\s*[\"'“”].*?[\"'“”]\s*$",
            "",
            party_two
        ).strip()

        cleaned["Party Two"] = party_two
    else:
        cleaned["Party Two"] = None


    return cleaned

## 4. Evaluation


In [19]:
def evaluate(train_df, predictions_df):

    truth = train_df.copy()
    pred = predictions_df.copy()

    # --------------------------------------------------
    # Ground-truth column mapping
    # --------------------------------------------------

    truth_column_map = {
        "Agreement Value": "Aggrement Value",
        "Agreement Start Date": "Aggrement Start Date",
        "Agreement End Date": "Aggrement End Date",
        "Renewal Notice (Days)": "Renewal Notice (Days)",
        "Party One": "Party One",
        "Party Two": "Party Two"
    }

    # --------------------------------------------------
    # Keep only the required columns
    # --------------------------------------------------

    truth = truth[
        ["File Name"] + [
            truth_column_map[field]
            for field in SCHEMA_FIELDS
        ]
    ].copy()

    # --------------------------------------------------
    # Rename ground-truth columns
    # --------------------------------------------------

    truth = truth.rename(
        columns={
            truth_column_map[field]: f"{field}_truth"
            for field in SCHEMA_FIELDS
        }
    )

    # --------------------------------------------------
    # Rename prediction columns
    # --------------------------------------------------

    pred = pred.rename(
        columns={
            field: f"{field}_pred"
            for field in SCHEMA_FIELDS
        }
    )

    pred = pred[
        ["File Name"] + [
            f"{field}_pred"
            for field in SCHEMA_FIELDS
        ]
    ].copy()

    # --------------------------------------------------
    # Normalize file names before merging
    # --------------------------------------------------

    truth["File Name"] = (
        truth["File Name"]
        .astype(str)
        .str.strip()
    )

    pred["File Name"] = (
        pred["File Name"]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------
    # Merge ground truth + predictions
    # --------------------------------------------------

    merged_df = pd.merge(
        truth,
        pred,
        on="File Name",
        how="left"
    )

    # --------------------------------------------------
    # Exact-match evaluation
    # --------------------------------------------------

    evaluation_results = []

    for field in SCHEMA_FIELDS:

        truth_col = f"{field}_truth"
        pred_col = f"{field}_pred"

        eligible = merged_df[truth_col].notna()

        correct = (
            merged_df.loc[eligible, truth_col]
            .astype(str)
            .str.strip()
            ==
            merged_df.loc[eligible, pred_col]
            .astype(str)
            .str.strip()
        ).sum()

        eligible_count = int(eligible.sum())

        recall = (
            correct / eligible_count * 100
            if eligible_count > 0
            else 0
        )

        evaluation_results.append({
            "Field": field,
            "Correct": int(correct),
            "Eligible": eligible_count,
            "Exact-match recall (%)": round(recall, 2)
        })

    evaluation_df = pd.DataFrame(evaluation_results)

    return evaluation_df, merged_df

In [21]:
evaluation_df, merged_df = evaluate(
    train_df,
    predictions_df
)

evaluation_df.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False
)

display(evaluation_df)

,Field,Correct,Eligible,Exact-match recall (%)
0,Agreement Value,4,10,40.0
1,Agreement Start Date,5,10,50.0
2,Agreement End Date,2,10,20.0
3,Renewal Notice (Days),0,9,0.0
4,Party One,4,10,40.0
5,Party Two,4,10,40.0


In [12]:
# ============================================================
# TRAINING SET EVALUATION
# ============================================================

# Ground-truth columns in train_df use "Aggrement" spelling.
# Rename them only in a copy for evaluation.
truth_df = train_df.copy()

truth_df = truth_df.rename(columns={
    "Aggrement Value": "Agreement Value",
    "Aggrement Start Date": "Agreement Start Date",
    "Aggrement End Date": "Agreement End Date"
})

# Keep only the columns required for evaluation
truth_df = truth_df[
    ["File Name"] + SCHEMA_FIELDS
].copy()

# Merge ground truth with predictions
evaluation_df = truth_df.merge(
    predictions_df,
    on="File Name",
    how="left",
    suffixes=("_truth", "_pred")
)

print("Evaluation rows:", len(evaluation_df))
print("Evaluation columns:")
print(evaluation_df.columns.tolist())

display(evaluation_df)

Evaluation rows: 10
Evaluation columns:
['File Name', 'Agreement Value_truth', 'Agreement Start Date_truth', 'Agreement End Date_truth', 'Renewal Notice (Days)_truth', 'Party One_truth', 'Party Two_truth', 'Split', 'Agreement Value_pred', 'Agreement Start Date_pred', 'Agreement End Date_pred', 'Renewal Notice (Days)_pred', 'Party One_pred', 'Party Two_pred', 'Extraction Status']


,File Name,Agreement Value_truth,Agreement Start Date_truth,Agreement End Date_truth,Renewal Notice (Days)_truth,Party One_truth,Party Two_truth,Split,Agreement Value_pred,Agreement Start Date_pred,Agreement End Date_pred,Renewal Notice (Days)_pred,Party One_pred,Party Two_pred,Extraction Status
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,train,6500,20.05.2007,20.05.2008,3,ANTONIO LEVY S. INGLES. JR. and/or MARY ROSE C...,GERALDINE O. GALINATO,success
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,20.05.2007,20.05.2008,15.0,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",GERALDINE Q. GALINATO,train,NaN,NaN,NaN,NaN,NaN,NaN,"error: HTTPConnectionPool(host='localhost', po..."
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90.0,MR.K.Kuttan,P.M. Narayana Namboodri,train,4000,05.12.2008,16.12.2009,90,MR.K.Kuttan,P.M. Narayana Namboodri 'Laxmi Leela',success
3,24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60.0,Hanumaiah,Vishal Bhardwaj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30.0,Balaji.R,Kartheek R,train,40000,01.05.2010,11.05.2011,30,Balaji.R,Kartheek R,success
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,3000,20.09.2010,19.07.2011,NaN,M.V.V. VIJAYA SHANKAR,MADDIREDDY BHARGAVA REDDY,train,9979,NaN,NaN,NaN,M. V. V« VI JAY A SHANKAR,MADDIREDDY BHARGAVA REDDY,success
6,47854715-RENTAL-AGREEMENT,9000,01.04.2010,31.02.2011,60.0,P C MATHEW,L GOPINATH,train,9000,01.04.2010,NaN,60,P C MATHEW,L GOPINATH,success
7,50070534-RENTAL-AGREEMENT,10000,01.04.2010,30.03.2011,90.0,P. JohnsonRavikumar,Saravanan BV,train,10000,01.04.2010,30.03.2011,90,P. JohnsonRavikumar,Saravanan BV,success
8,54770958-Rental-Agreement,8000,01.04.2011,31.03.2012,90.0,K. Parthasarathy,Veerabrahmam Bathini,train,5500,02.04.2011,02.04.2012,60,Mrs.Asha Ramesh & Mr.Ramesh.K.N,Mrs.Sadasivuni Deepti & Mr.Sadasivuni Kiran,success
9,54945838-Rental-Agreement,5500,21.04.2011,19.02.2012,60.0,Asha Ramesh & Ramesh K.N,Sadasivuni Deepthi & Sadasivuni Kiran,train,8000,01.04.2011,01.04.2013,90,K. Parthasarathy,Veerabrahmam Bathini,success


In [13]:
# ============================================================
# EXACT-MATCH RECALL
# ============================================================

evaluation_results = []

for field in SCHEMA_FIELDS:

    truth_col = f"{field}_truth"
    pred_col = f"{field}_pred"

    eligible = evaluation_df[truth_col].notna()

    correct = (
        evaluation_df.loc[eligible, truth_col].astype(str).str.strip()
        ==
        evaluation_df.loc[eligible, pred_col].astype(str).str.strip()
    ).sum()

    eligible_count = eligible.sum()

    recall = (
        correct / eligible_count * 100
        if eligible_count > 0
        else 0
    )

    evaluation_results.append({
        "Field": field,
        "Correct": correct,
        "Eligible": eligible_count,
        "Exact-match recall (%)": round(recall, 2)
    })

evaluation_summary_df = pd.DataFrame(evaluation_results)

display(evaluation_summary_df)

,Field,Correct,Eligible,Exact-match recall (%)
0,Agreement Value,4,10,40.0
1,Agreement Start Date,5,10,50.0
2,Agreement End Date,2,10,20.0
3,Renewal Notice (Days),0,9,0.0
4,Party One,4,10,40.0
5,Party Two,4,10,40.0


## 5. Evaluation

In [20]:
evaluation_df, merged_df = evaluate(train_df, predictions_df)
evaluation_df.to_csv(OUTPUT_DIR / "evaluation_summary.csv", index=False)
display(evaluation_df)

,Field,Correct,Eligible,Exact-match recall (%)
0,Agreement Value,4,10,40.0
1,Agreement Start Date,5,10,50.0
2,Agreement End Date,2,10,20.0
3,Renewal Notice (Days),0,9,0.0
4,Party One,4,10,40.0
5,Party Two,4,10,40.0


## 6. Failure analysis

In [24]:
def failure_cases(merged_df):

    failures = {}

    for field in SCHEMA_FIELDS:

        truth_col = f"{field}_truth"
        pred_col = f"{field}_pred"

        eligible = merged_df[truth_col].notna()

        comparison = (
            merged_df[truth_col]
            .astype(str)
            .str.strip()
            ==
            merged_df[pred_col]
            .astype(str)
            .str.strip()
        )

        failures[field] = merged_df.loc[
            eligible & ~comparison,
            [
                "File Name",
                truth_col,
                pred_col
            ]
        ].copy()

    return failures

In [25]:
failures = failure_cases(merged_df)

for field, table in failures.items():

    print(f"\n==================== {field} ====================")

    display(table)


==================== Agreement Value ====================


,File Name,Agreement Value_truth,Agreement Value_pred
1,6683129-House-Rental-Contract-Geraldine-Galina...,6500,NaN
3,24158401-Rental-Agreement,12000,NaN
4,36199312-Rental-Agreement,3800,40000
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,3000,9979
8,54770958-Rental-Agreement,8000,5500
9,54945838-Rental-Agreement,5500,8000



==================== Agreement Start Date ====================


,File Name,Agreement Start Date_truth,Agreement Start Date_pred
1,6683129-House-Rental-Contract-Geraldine-Galina...,20.05.2007,NaN
3,24158401-Rental-Agreement,01.04.2008,NaN
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,20.09.2010,NaN
8,54770958-Rental-Agreement,01.04.2011,02.04.2011
9,54945838-Rental-Agreement,21.04.2011,01.04.2011



==================== Agreement End Date ====================


...

,File Name,Renewal Notice (Days)_truth,Renewal Notice (Days)_pred
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,15.0,3
1,6683129-House-Rental-Contract-Geraldine-Galina...,15.0,NaN
2,18325926-Rental-Agreement-1,90.0,90
3,24158401-Rental-Agreement,60.0,NaN
4,36199312-Rental-Agreement,30.0,30
6,47854715-RENTAL-AGREEMENT,60.0,60
7,50070534-RENTAL-AGREEMENT,90.0,90
8,54770958-Rental-Agreement,90.0,60
9,54945838-Rental-Agreement,60.0,90



==================== Party One ====================


,File Name,Party One_truth,Party One_pred
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",ANTONIO LEVY S. INGLES. JR. and/or MARY ROSE C...
1,6683129-House-Rental-Contract-Geraldine-Galina...,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C...",NaN
3,24158401-Rental-Agreement,Hanumaiah,NaN
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,M.V.V. VIJAYA SHANKAR,M. V. V« VI JAY A SHANKAR
8,54770958-Rental-Agreement,K. Parthasarathy,Mrs.Asha Ramesh & Mr.Ramesh.K.N
9,54945838-Rental-Agreement,Asha Ramesh & Ramesh K.N,K. Parthasarathy



==================== Party Two ====================


,File Name,Party Two_truth,Party Two_pred
0,6683127-House-Rental-Contract-GERALDINE-GALINA...,GERALDINE Q. GALINATO,GERALDINE O. GALINATO
1,6683129-House-Rental-Contract-Geraldine-Galina...,GERALDINE Q. GALINATO,NaN
2,18325926-Rental-Agreement-1,P.M. Narayana Namboodri,P.M. Narayana Namboodri 'Laxmi Leela'
3,24158401-Rental-Agreement,Vishal Bhardwaj,NaN
8,54770958-Rental-Agreement,Veerabrahmam Bathini,Mrs.Sadasivuni Deepti & Mr.Sadasivuni Kiran
9,54945838-Rental-Agreement,Sadasivuni Deepthi & Sadasivuni Kiran,Veerabrahmam Bathini
